# BCH(7,4) 완전한 예제 - 신드롬부터 Chien Search까지

이 노트북은 BCH(7,4) 코드를 사용하여 BCH 디코딩의 전체 과정을 단계별로 학습합니다.

## 목차
1. [BCH(7,4) 설정](#1.-BCH(7,4)-설정)
2. [GF(2³) Power Table](#2.-GF(2³)-Power-Table)
3. [예제 1: 위치 5에 에러](#3.-예제-1:-위치-5에-에러)
4. [예제 2: 위치 2에 에러](#4.-예제-2:-위치-2에-에러)
5. [예제 3: 위치 0에 에러](#5.-예제-3:-위치-0에-에러)
6. [Chien Search 원리 설명](#6.-Chien-Search-원리-설명)
7. [인덱싱 컨벤션](#7.-인덱싱-컨벤션)

In [ ]:
# 필요한 모듈 임포트
import sys
sys.path.append('..')

from bch_learning import (
    GaloisField,
    BCHCode,
    BerlekampMassey,
    ChienSearch
)
from bch_learning.syndrome_calculator import SyndromeCalculator
import numpy as np

## 1. BCH(7,4) 설정

### 파라미터

- **n = 7**: 코드워드 길이
- **k = 4**: 정보 비트 수
- **t = 1**: 정정 가능한 에러 개수
- **GF(2³)**: 유한체
- **Primitive polynomial**: α³ = α + 1 (이진: 1011)

### 인덱싱

코드워드 표현:
```
r = (r₀, r₁, r₂, r₃, r₄, r₅, r₆)
r(x) = r₀ + r₁x + r₂x² + ... + r₆x⁶
```

**중요**: 위치 i의 에러는 Error locator X = α^i로 표현됩니다.

In [ ]:
# GF(2³) 생성
gf = GaloisField(3)  # m=3 → GF(2³)

print("="*80)
print("BCH(7,4) 코드 설정")
print("="*80)
print(f"유한체: {gf}")
print(f"원시 다항식: {bin(gf.primitive_poly)} (α³ = α + 1)")
print(f"코드 길이: n = {gf.order}")
print("="*80)

## 2. GF(2³) Power Table

이 테이블은 GF(2³)의 모든 원소를 다양한 표현으로 보여줍니다.

In [ ]:
# Power table 출력
gf.print_table()

# 추가 설명
print("\n주요 관계식:")
print("  α³ = α + 1 (원시 다항식)")
print("  α⁷ = α⁰ = 1 (order = 7)")
print("  α⁻¹ = α⁶")
print("  α⁻⁵ = α² (∵ -5 mod 7 = 2)")

## 3. 예제 1: 위치 5에 에러

### 설정

- **전송**: c = (0,0,0,0,0,0,0) (all-zero 코드워드)
- **에러**: e = (0,0,0,0,0,1,0) → 위치 5에 1
- **수신**: r = (0,0,0,0,0,1,0)

### 이론적 예상

위치 5에 에러가 있으므로:
```
e(x) = x⁵
S₁ = e(α) = α⁵
S₂ = e(α²) = (α²)⁵ = α¹⁰ = α³
```

또는 S₂ = (S₁)² = (α⁵)² = α¹⁰ = α³

In [ ]:
print("="*80)
print("예제 1: 위치 5에 에러")
print("="*80)

# 수신 벡터
received_ex1 = [0, 0, 0, 0, 0, 1, 0]  # r = (r₀, r₁, r₂, r₃, r₄, r₅, r₆)
print(f"\n수신 벡터: r = ({''.join(map(str, received_ex1))})")
print(f"인덱싱: r = (r₀, r₁, r₂, r₃, r₄, r₅, r₆)")
print(f"에러 위치: 5 (r₅ = 1)")

### 3.1. 신드롬 계산

In [ ]:
print("\n" + "-"*80)
print("[1단계] 신드롬 계산")
print("-"*80)

calc = SyndromeCalculator(gf, t=1, verbose=False)
result_ex1 = calc.compute_by_evaluation(received_ex1)

s1 = result_ex1.syndromes[0]
s2 = result_ex1.syndromes[1]

print(f"\nS₁ = {s1}")
print(f"S₂ = {s2}")

# 검증
expected_s1 = gf.alpha(5)
expected_s2 = gf.alpha(3)  # α¹⁰ = α³

print(f"\n✓ 검증:")
print(f"  S₁ = α⁵? {s1 == expected_s1}")
print(f"  S₂ = α³? {s2 == expected_s2}")
print(f"  S₂ = (S₁)²? {s2 == s1**2}")

# 상세 계산
print(f"\n상세 계산:")
print(f"  S₁ = Σ(i=0 to 6) rᵢ·αⁱ")
print(f"     = r₅·α⁵")
print(f"     = 1·α⁵")
print(f"     = α⁵")
print(f"")
print(f"  S₂ = Σ(i=0 to 6) rᵢ·α²ⁱ")
print(f"     = r₅·α¹⁰")
print(f"     = α¹⁰ mod 7")
print(f"     = α³")
print(f"")
print(f"  또는: S₂ = (S₁)² = (α⁵)² = α¹⁰ = α³ ✓")

### 3.2. Berlekamp-Massey 알고리즘

In [ ]:
print("\n" + "-"*80)
print("[2단계] Berlekamp-Massey 알고리즘")
print("-"*80)

bm_ex1 = BerlekampMassey(result_ex1.syndromes, verbose=False)
elp_ex1, iterations_ex1 = bm_ex1.run()

print(f"\n반복 0:")
print(f"  Δ⁽⁰⁾ = S₁ = {s1} ≠ 0")
print(f"  2L⁽⁻¹⁾ = 0 ≤ n = 0 ✓")
print(f"  → 길이 갱신")
print(f"  Λ⁽⁰⁾(x) = 1 + {s1}·x")
print(f"  L⁽⁰⁾ = 1")

print(f"\n반복 1:")
print(f"  Δ⁽¹⁾ = S₂ + λ₁⁽⁰⁾·S₁")
print(f"       = {s2} + {s1}·{s1}")
print(f"       = {s2} + {s1 * s1}")
print(f"       = {s2 + s1 * s1}")
print(f"  Δ = 0 → 업데이트 없음")

print(f"\n최종 오류 위치 다항식:")
print(f"  Λ(x) = {elp_ex1[0]} + {elp_ex1[1]}·x")
print(f"  예상: Λ(x) = 1 + α⁵·x")
print(f"  일치: {elp_ex1[0] == gf.one() and elp_ex1[1] == gf.alpha(5)}")

### 3.3. Chien Search

#### 개념 복습

- 에러 위치 i → Error locator: X = α^i
- ELP: Λ(x) = 1 + X·x (단일 에러)
- ELP의 근: X⁻¹ = α^(-i)

#### 우리 예시

```
Λ(x) = 1 + α⁵·x
→ Error locator: X = α⁵ (에러 위치 5)
→ ELP의 근: X⁻¹ = α⁻⁵ = α² (∵ -5 mod 7 = 2)
```

In [ ]:
print("\n" + "-"*80)
print("[3단계] Chien Search - 근 찾기")
print("-"*80)

print(f"\nΛ(x) = 1 + {elp_ex1[1]}·x")
print(f"\nα⁰부터 α⁶까지 대입:")
print(f"\n{'α^j':<8} {'Λ(α^j) = 1 + α⁵·α^j':<25} {'계산':<20} {'결과':<10}")
print("-"*70)

root_found = None
for j in range(7):
    alpha_j = gf.alpha(j)
    value = elp_ex1[0] + elp_ex1[1] * alpha_j
    
    # 이진수 계산 표시
    val1 = elp_ex1[0].poly
    val2 = (elp_ex1[1] * alpha_j).poly
    result_bin = val1 ^ val2
    
    is_root = value.is_zero()
    result_str = "= 0 ✓" if is_root else "≠ 0"
    
    # α⁵ + α⁷의 경우 특별 표시
    if j == 2:
        calc_str = f"{val1:03b} + {val2:03b} = {result_bin:03b}"
    else:
        calc_str = f"{val1:03b} + {val2:03b} = {result_bin:03b}"
    
    print(f"α^{j:<6} {'1 + ' + str(elp_ex1[1] * alpha_j):<25} {calc_str:<20} {result_str:<10}")
    
    if is_root:
        root_found = j

print(f"\n근: α^{root_found}")

### 3.4. 에러 위치 복구

#### 공식

```
근 = α^j → 에러 위치 = n - j (mod n)
```

#### 왜 이렇게 되는가?

```
에러 위치 i → Error locator X = α^i
ELP의 근 = X⁻¹ = α^(-i) = α^(n-i)  (∵ -i mod n = n-i)
Chien search에서 찾은 근 = α^j
→ j = n - i
→ i = n - j
```

In [ ]:
print("\n" + "-"*80)
print("[4단계] 에러 위치 복구")
print("-"*80)

error_pos_calculated = (7 - root_found) % 7

print(f"\n공식: 에러 위치 = n - j (mod n)")
print(f"")
print(f"계산:")
print(f"  근 = α^{root_found}")
print(f"  j = {root_found}")
print(f"  에러 위치 = 7 - {root_found} = {error_pos_calculated}")

print(f"\n검증:")
print(f"  Error locator: X = α⁵")
print(f"  ELP의 근: X⁻¹ = α⁻⁵ = α^(7-5) = α² ✓")
print(f"  실제 에러 위치: 5 ✓")

# 실제 Chien Search 실행
cs_ex1 = ChienSearch(elp_ex1, 7, verbose=False)
found_errors_ex1 = cs_ex1.search()

print(f"\nChien Search 결과: {found_errors_ex1}")
print(f"예상: [5]")
print(f"일치: {found_errors_ex1 == [5]}")

## 4. 예제 2: 위치 2에 에러

### 설정

- **수신**: r = (0,0,1,0,0,0,0)
- **에러 위치**: 2 (r₂ = 1)

### 예상

```
S₁ = r₂·α² = α²
Λ(x) = 1 + α²·x
근: α⁵ (∵ Λ(α⁵) = 1 + α²·α⁵ = 1 + α⁷ = 1 + 1 = 0)
에러 위치 = 7 - 5 = 2 ✓
```

In [ ]:
print("\n" + "="*80)
print("예제 2: 위치 2에 에러")
print("="*80)

received_ex2 = [0, 0, 1, 0, 0, 0, 0]
print(f"\n수신: {''.join(map(str, received_ex2))}")
print(f"에러 위치: 2")

# 신드롬
result_ex2 = calc.compute_by_evaluation(received_ex2)
s1_ex2 = result_ex2.syndromes[0]

print(f"\nS₁ = {s1_ex2} (예상: α²)")
print(f"일치: {s1_ex2 == gf.alpha(2)}")

# BM
bm_ex2 = BerlekampMassey(result_ex2.syndromes, verbose=False)
elp_ex2, _ = bm_ex2.run()

print(f"\nΛ(x) = {elp_ex2[0]} + {elp_ex2[1]}·x")
print(f"예상: Λ(x) = 1 + α²·x")

# Chien Search
print(f"\nChien Search:")
for j in range(7):
    val = elp_ex2[0] + elp_ex2[1] * gf.alpha(j)
    if val.is_zero():
        print(f"  근: α^{j}")
        print(f"  에러 위치 = 7 - {j} = {7-j} ✓")
        break

cs_ex2 = ChienSearch(elp_ex2, 7, verbose=False)
found_ex2 = cs_ex2.search()
print(f"\n결과: {found_ex2}")
print(f"예상: [2]")
print(f"일치: {found_ex2 == [2]}")

## 5. 예제 3: 위치 0에 에러

### 설정

- **수신**: r = (1,0,0,0,0,0,0)
- **에러 위치**: 0 (r₀ = 1)

### 예상

```
S₁ = r₀·α⁰ = 1
Λ(x) = 1 + 1·x = 1 + x
근: α⁰ = 1 (∵ Λ(α⁰) = 1 + α⁰ = 1 + 1 = 0)
에러 위치 = 7 - 0 = 7 mod 7 = 0 ✓
```

In [ ]:
print("\n" + "="*80)
print("예제 3: 위치 0에 에러")
print("="*80)

received_ex3 = [1, 0, 0, 0, 0, 0, 0]
print(f"\n수신: {''.join(map(str, received_ex3))}")
print(f"에러 위치: 0")

# 신드롬
result_ex3 = calc.compute_by_evaluation(received_ex3)
s1_ex3 = result_ex3.syndromes[0]

print(f"\nS₁ = {s1_ex3} (예상: 1 = α⁰)")
print(f"일치: {s1_ex3 == gf.one()}")

# BM
bm_ex3 = BerlekampMassey(result_ex3.syndromes, verbose=False)
elp_ex3, _ = bm_ex3.run()

print(f"\nΛ(x) = {elp_ex3[0]} + {elp_ex3[1]}·x")
print(f"예상: Λ(x) = 1 + x")

# Chien Search
print(f"\nChien Search:")
for j in range(7):
    val = elp_ex3[0] + elp_ex3[1] * gf.alpha(j)
    if val.is_zero():
        print(f"  근: α^{j}")
        print(f"  에러 위치 = 7 - {j} mod 7 = {(7-j)%7} ✓")
        break

cs_ex3 = ChienSearch(elp_ex3, 7, verbose=False)
found_ex3 = cs_ex3.search()
print(f"\n결과: {found_ex3}")
print(f"예상: [0]")
print(f"일치: {found_ex3 == [0]}")

## 6. Chien Search 원리 설명

### Error Locator Polynomial의 정의

GF(2)에서 단일 에러의 경우:

```
Λ(x) = (1 + Xᵢ·x)
```

여기서 Xᵢ = α^i는 에러 위치 i의 Error locator입니다.

### 근의 성질

ELP의 근을 x = β라 하면:

```
Λ(β) = 0
1 + Xᵢ·β = 0
Xᵢ·β = 1
β = Xᵢ⁻¹
```

따라서 **ELP의 근 = Error locator의 역원**

### 에러 위치 복구 공식

```
에러 위치 i → Error locator X = α^i
ELP의 근 = X⁻¹ = α^(-i) = α^(n-i)  (GF(2^m)에서 α^n = α^0)
```

Chien Search에서 α^j를 대입하여 근을 찾았다면:

```
α^j = α^(n-i)
j = n - i (mod n)
i = n - j (mod n)
```

### 우리 구현 방식

우리 코드는 각 위치 i에 대해 α^(-i)를 대입합니다:

```python
for i in range(n):
    alpha_inv_i = field.alpha(-i)  # = α^(n-i)
    if Λ(alpha_inv_i) == 0:
        error_position = i
```

이것은 다음과 같이 작동합니다:

```
위치 i에 에러 → Error locator X = α^i
Λ(α^(-i)) = Λ(X⁻¹) = 1 + X·X⁻¹ = 1 + 1 = 0 ✓
```

### 두 방법의 동등성

**방법 1** (예제 문서):
```
α^0, α^1, ..., α^(n-1)을 차례로 대입
근 α^j 발견 → 에러 위치 = n - j
```

**방법 2** (우리 구현):
```
위치 0, 1, ..., n-1에 대해 α^(-i) 대입
Λ(α^(-i)) = 0 → 에러 위치 = i
```

두 방법은 완전히 동등합니다:
```
방법 1에서 j번째 검사 = 방법 2에서 (n-j)번째 검사
```

In [ ]:
# 두 방법 비교 시연
print("="*80)
print("두 방법의 동등성 검증")
print("="*80)

# 예제 1의 ELP 사용: Λ(x) = 1 + α⁵·x
print(f"\nΛ(x) = 1 + α⁵·x (위치 5에 에러)")

print(f"\n방법 1: α^0부터 α^6까지 대입")
for j in range(7):
    val = elp_ex1[0] + elp_ex1[1] * gf.alpha(j)
    if val.is_zero():
        print(f"  근 발견: α^{j}")
        print(f"  에러 위치 = 7 - {j} = {7-j}")

print(f"\n방법 2: 각 위치 i에 대해 α^(-i) 대입")
for i in range(7):
    alpha_inv_i = gf.alpha(-i)
    val = elp_ex1[0] + elp_ex1[1] * alpha_inv_i
    if val.is_zero():
        print(f"  위치 {i}: Λ(α^(-{i})) = Λ(α^{(-i)%7}) = 0")
        print(f"  에러 위치 = {i}")

print(f"\n✓ 두 방법 모두 에러 위치 5를 찾음!")

## 7. 인덱싱 컨벤션

### 우리가 사용하는 방식 (표준)

```
코드워드: r = (r₀, r₁, ..., r_{n-1})
다항식: r(x) = r₀ + r₁x + ... + r_{n-1}x^{n-1}
에러 위치 i → Error locator X = α^i
ELP의 근 α^j → 에러 위치 = n - j (방법 1)
또는 위치 i에 α^(-i) 대입 (방법 2)
```

### 일부 교재의 역순 방식

```
코드워드: r = (r_{n-1}, r_{n-2}, ..., r₀)
다항식: r(x) = r_{n-1} + r_{n-2}x + ... + r₀x^{n-1}
에러 위치 i → Error locator X = α^{n-1-i}
ELP의 근 α^j → 에러 위치 = j
```

### 중요

**항상 사용하는 컨벤션을 명확히 하세요!**

우리 구현은 **표준 컨벤션**을 사용합니다:
- 위치 i의 비트 = r_i
- Error locator = α^i
- Chien Search: 위치 i에 α^(-i) 대입

## 요약

### BCH(7,4) 디코딩 흐름

1. **신드롬 계산**: S_i = R(α^i)
2. **Berlekamp-Massey**: 신드롬 → Λ(x)
3. **Chien Search**: Λ(x)의 근 찾기 → 에러 위치
4. **에러 정정**: r_i ⊕ 1

### 검증된 예제

- ✓ 위치 5에 에러: S₁ = α⁵, Λ(x) = 1 + α⁵x, 근 = α², 위치 = 5
- ✓ 위치 2에 에러: S₁ = α², Λ(x) = 1 + α²x, 근 = α⁵, 위치 = 2
- ✓ 위치 0에 에러: S₁ = 1, Λ(x) = 1 + x, 근 = α⁰, 위치 = 0

### 핵심 공식

```
에러 위치 i → Error locator X = α^i
ELP의 근 = X⁻¹ = α^(-i) = α^(n-i)
Chien Search: Λ(α^(-i)) = 0 → 에러 위치 i
```